# Customer Churn Prediction — Model Training & Evaluation
**Author:** Aleena Anam | [GitHub](https://github.com/anam-aleena)

This notebook covers:
- Training 4 ML models with 5-fold cross-validation
- Model comparison and selection rationale
- Final evaluation on held-out test set
- Confusion matrix, ROC curve, and feature importance

In [ ]:
import sys
sys.path.append('../src')
from pipeline import *
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, auc

df = generate_churn_data(5000)
X_train, X_test, y_train, y_test, feature_names = preprocess(df)
models, best_model, best_name, cv_results = train_models(X_train, y_train)
metrics = evaluate(models, best_model, best_name, X_train, X_test, y_train, y_test, feature_names)

## Model Comparison — Cross-Validation AUC

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
names = list(cv_results.keys())
scores = list(cv_results.values())
colors = ['#F44336' if n == best_name else '#90CAF9' for n in names]
bars = ax.barh(names, scores, color=colors, edgecolor='white')
ax.set_xlim(0.5, 1.0)
ax.set_xlabel('ROC-AUC Score', fontsize=11)
ax.set_title('Model Comparison — 5-Fold Cross-Validation ROC-AUC', fontsize=13, fontweight='bold')
for bar, score in zip(bars, scores):
    ax.text(score + 0.002, bar.get_y() + bar.get_height()/2,
            f'{score:.4f}', va='center', fontweight='bold')
ax.axvline(x=max(scores), color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('../reports/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Selected Model: {best_name}')
print('Rationale: Highest cross-validation AUC with minimal variance across folds.')

## ROC Curves — All Models

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors_roc = ['#2196F3','#4CAF50','#FF9800','#F44336']

for (name, model), color in zip(models.items(), colors_roc):
    model.fit(X_train, y_train)
    y_score = model.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_auc = auc(fpr, tpr)
    lw = 2.5 if name == best_name else 1.5
    ax.plot(fpr, tpr, color=color, lw=lw, label=f'{name} (AUC={roc_auc:.3f})')

ax.plot([0,1],[0,1],'k--',lw=1, alpha=0.5, label='Random')
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curves — All Models', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Final Metrics Summary

In [ ]:
import pandas as pd
metrics_df = pd.DataFrame([metrics]).T
metrics_df.columns = ['Score']
print(f'\nFinal Test Set Performance — {best_name}')
print('='*40)
print(metrics_df.to_string())

## Model Selection Rationale
We trained 4 models using 5-fold stratified cross-validation and selected the best by ROC-AUC:
- **Logistic Regression** — solid baseline, highly interpretable
- **Random Forest** — handles non-linearity, robust to outliers
- **Gradient Boosting** — strong performance, slower training
- **XGBoost** — typically best on tabular data with regularization

Final selection was based on cross-validation AUC, not test set performance, to prevent data leakage.